# Pandas 101

This is an optional notebook to get you up to speed with [pandas](https://pandas.pydata.org/docs/) in case you are new to it or need a refresher. We use pandas throughout this module to load, inspect, clean, and join data. The [official 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html) guide is a good deeper dive once you've been through this.

We'll work with a small sales dataset (`data/sales.csv`) so everything below runs against real data rather than toy examples.

## DataFrames and Series

Pandas gives us two main data structures:

- A **Series** is a single column of data (with an index).
- A **DataFrame** is a table made up of rows and columns - think of it as a collection of Series that share the same index.

By convention, we import pandas as `pd`:

In [1]:
import pandas as pd

## Reading Data

We can load a CSV file straight into a DataFrame with `read_csv()`:

In [2]:
df = pd.read_csv('data/sales.csv')
type(df)

pandas.DataFrame

## Inspecting a DataFrame

Once we have loaded some data, the first thing to do is look at it. `.shape` tells us the number of rows and columns:

In [3]:
df.shape

(32718, 9)

`.head()` shows the first few rows (5 by default). We can pass a number to see more or fewer:

In [4]:
df.head(3)

,SalesOrderNumber,SalesOrderLineNumber,OrderDate,CustomerName,EmailAddress,Item,Quantity,UnitPrice,TaxAmount
0,SO43701,1,2019-07-01,Christy Zhu,christy12@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
1,SO43704,1,2019-07-01,Julio Ruiz,julio1@adventure-works.com,"Mountain-100 Black, 48",1,3374.99,269.9992
2,SO43705,1,2019-07-01,Curtis Lu,curtis9@adventure-works.com,"Mountain-100 Silver, 38",1,3399.99,271.9992


`.dtypes` shows the data type pandas has inferred for each column. This matters - if a numeric column has been read in as text (`object`), calculations on it will fail or behave unexpectedly:

In [5]:
df.dtypes

SalesOrderNumber            str
SalesOrderLineNumber      int64
OrderDate                   str
CustomerName                str
EmailAddress                str
Item                        str
Quantity                  int64
UnitPrice               float64
TaxAmount               float64
dtype: object

`.info()` combines several of these into one summary, including how many non-null values are in each column:

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32718 entries, 0 to 32717
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   SalesOrderNumber      32718 non-null  str    
 1   SalesOrderLineNumber  32718 non-null  int64  
 2   OrderDate             32718 non-null  str    
 3   CustomerName          32718 non-null  str    
 4   EmailAddress          32718 non-null  str    
 5   Item                  32718 non-null  str    
 6   Quantity              32718 non-null  int64  
 7   UnitPrice             32718 non-null  float64
 8   TaxAmount             32718 non-null  float64
dtypes: float64(2), int64(2), str(5)
memory usage: 2.2 MB


## Selecting Columns

We can select a single column using square brackets with the column name. This returns a **Series**:

In [7]:
df['Item']

0               Mountain-100 Silver, 44
1                Mountain-100 Black, 48
2               Mountain-100 Silver, 38
3                    Road-650 Black, 62
4                      Road-150 Red, 62
                      ...              
32713               Patch Kit/8 Patches
32714             Road-550-W Yellow, 48
32715    Short-Sleeve Classic Jersey, S
32716                Mountain Tire Tube
32717               Patch Kit/8 Patches
Name: Item, Length: 32718, dtype: str

To select more than one column, pass a list of names. This returns a **DataFrame** (note the double square brackets - the outer `[]` is selecting, the inner `[]` is the list of column names):

In [8]:
df[['Item', 'UnitPrice']]

,Item,UnitPrice
0,"Mountain-100 Silver, 44",3399.9900
1,"Mountain-100 Black, 48",3374.9900
2,"Mountain-100 Silver, 38",3399.9900
3,"Road-650 Black, 62",699.0982
4,"Road-150 Red, 62",3578.2700
...,...,...
32713,Patch Kit/8 Patches,2.2900
32714,"Road-550-W Yellow, 48",1120.4900
32715,"Short-Sleeve Classic Jersey, S",53.9900
32716,Mountain Tire Tube,4.9900


## Selecting Rows

The most common way to select rows is with a **boolean mask**. First, a comparison on a column gives us a Series of `True`/`False` values:

In [9]:
df['UnitPrice'] > 3000

0         True
1         True
2         True
3        False
4         True
         ...  
32713    False
32714    False
32715    False
32716    False
32717    False
Name: UnitPrice, Length: 32718, dtype: bool

Passing that mask back into `df[...]` keeps only the rows where the value was `True`:

In [10]:
df[df['UnitPrice'] > 3000]

,SalesOrderNumber,SalesOrderLineNumber,OrderDate,CustomerName,EmailAddress,Item,Quantity,UnitPrice,TaxAmount
0,SO43701,1,2019-07-01,Christy Zhu,christy12@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
1,SO43704,1,2019-07-01,Julio Ruiz,julio1@adventure-works.com,"Mountain-100 Black, 48",1,3374.99,269.9992
2,SO43705,1,2019-07-01,Curtis Lu,curtis9@adventure-works.com,"Mountain-100 Silver, 38",1,3399.99,271.9992
4,SO43703,1,2019-07-01,Albert Alvarez,albert7@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
5,SO43697,1,2019-07-01,Cole Watson,cole1@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
...,...,...,...,...,...,...,...,...,...
2200,SO46603,1,2020-05-30,Clayton Sharma,clayton27@adventure-works.com,"Mountain-100 Silver, 42",1,3399.99,271.9992
2201,SO46599,1,2020-05-30,Sierra Parker,sierra7@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
2202,SO46598,1,2020-05-30,Evelyn Chandra,evelyn2@adventure-works.com,"Road-150 Red, 44",1,3578.27,286.2616
2203,SO46602,1,2020-05-30,Troy Martinez,troy19@adventure-works.com,"Road-150 Red, 48",1,3578.27,286.2616


We can combine conditions using `&` (and) and `|` (or). Each condition needs its own parentheses:

In [11]:
df[(df['UnitPrice'] > 3000) & (df['Quantity'] > 0)]

,SalesOrderNumber,SalesOrderLineNumber,OrderDate,CustomerName,EmailAddress,Item,Quantity,UnitPrice,TaxAmount
0,SO43701,1,2019-07-01,Christy Zhu,christy12@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
1,SO43704,1,2019-07-01,Julio Ruiz,julio1@adventure-works.com,"Mountain-100 Black, 48",1,3374.99,269.9992
2,SO43705,1,2019-07-01,Curtis Lu,curtis9@adventure-works.com,"Mountain-100 Silver, 38",1,3399.99,271.9992
4,SO43703,1,2019-07-01,Albert Alvarez,albert7@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
5,SO43697,1,2019-07-01,Cole Watson,cole1@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
...,...,...,...,...,...,...,...,...,...
2200,SO46603,1,2020-05-30,Clayton Sharma,clayton27@adventure-works.com,"Mountain-100 Silver, 42",1,3399.99,271.9992
2201,SO46599,1,2020-05-30,Sierra Parker,sierra7@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
2202,SO46598,1,2020-05-30,Evelyn Chandra,evelyn2@adventure-works.com,"Road-150 Red, 44",1,3578.27,286.2616
2203,SO46602,1,2020-05-30,Troy Martinez,troy19@adventure-works.com,"Road-150 Red, 48",1,3578.27,286.2616


## String Methods

Text columns have a `.str` accessor that gives us string operations applied across every row at once. For example, checking which rows contain a word:

In [12]:
df[df['Item'].str.contains('Mountain')].head(3)

,SalesOrderNumber,SalesOrderLineNumber,OrderDate,CustomerName,EmailAddress,Item,Quantity,UnitPrice,TaxAmount
0,SO43701,1,2019-07-01,Christy Zhu,christy12@adventure-works.com,"Mountain-100 Silver, 44",1,3399.99,271.9992
1,SO43704,1,2019-07-01,Julio Ruiz,julio1@adventure-works.com,"Mountain-100 Black, 48",1,3374.99,269.9992
2,SO43705,1,2019-07-01,Curtis Lu,curtis9@adventure-works.com,"Mountain-100 Silver, 38",1,3399.99,271.9992


Other common ones are `.str.upper()`, `.str.lower()`, and `.str.strip()` (removes leading/trailing whitespace) - useful when cleaning up messy, inconsistently-cased data.

## Missing Values

Our sales data happens to be complete, so let's build a small example with some gaps to show how pandas handles them:

In [13]:
example = pd.DataFrame({
    'product': ['Kettle', 'Toaster', 'Blender', 'Iron'],
    'price': [24.99, None, 18.50, None]
})
example

,product,price
0,Kettle,24.99
1,Toaster,NaN
2,Blender,18.50
3,Iron,NaN


`.isnull()` flags missing values; combined with `.sum()` we get a count per column:

In [14]:
example.isnull().sum()

product    0
price      2
dtype: int64

`.dropna()` removes any row containing a missing value:

In [15]:
example.dropna()

,product,price
0,Kettle,24.99
2,Blender,18.50


Alternatively, `.fillna()` replaces missing values rather than dropping the rows - here filling with the average price:

In [16]:
example['price'].fillna(example['price'].mean())

0    24.990
1    21.745
2    18.500
3    21.745
Name: price, dtype: float64

## Sorting

`.sort_values()` orders a DataFrame by one or more columns. Use `ascending=False` for highest first:

In [17]:
df.sort_values('UnitPrice', ascending=False).head(3)

,SalesOrderNumber,SalesOrderLineNumber,OrderDate,CustomerName,EmailAddress,Item,Quantity,UnitPrice,TaxAmount
1364,SO45508,1,2020-01-27,Roberto Sanz,roberto18@adventure-works.com,"Road-150 Red, 62",1,3578.27,286.2616
1804,SO46133,1,2020-04-03,Abigail Foster,abigail68@adventure-works.com,"Road-150 Red, 48",1,3578.27,286.2616
1070,SO45134,1,2019-12-11,Eduardo Perez,eduardo36@adventure-works.com,"Road-150 Red, 56",1,3578.27,286.2616


## Saving Data

Once we've selected or transformed the data we want, `.to_csv()` writes it back out. `index=False` stops pandas writing its own row-number index as an extra column:

In [18]:
expensive_items = df[df['UnitPrice'] > 3000]
expensive_items.to_csv('expensive_items.csv', index=False)
print(f'Saved {len(expensive_items)} rows')

Saved 1947 rows


##### END